# Environment Setup & Configuration

This notebook walks through setting up all the tools and dependencies needed to evaluate RAG (Retrieval-Augmented Generation) pipelines. By the end, you will have:

1. All Python packages installed (DeepEval, RAGAS, Qdrant, OpenAI, LangChain, Cohere, etc.)
2. Environment variables loaded and verified
3. Confirmed that each framework is operational
4. Run a quick "hello world" evaluation with both DeepEval and RAGAS

-
## 1. Install Dependencies

We install everything in one shot. The packages cover:

| Package | Purpose |
|---|---|
| `deepeval` | LLM evaluation framework (retriever + generator metrics) |
| `ragas` | RAG-specific evaluation framework |
| `qdrant-client` | Vector database client |
| `openai` | LLM provider for generation and embeddings |
| `langchain`, `langchain-openai`, `langchain-community` | Orchestration & integrations |
| `sentence-transformers` | Local embedding models & cross-encoder reranking |
| `python-dotenv` | Load `.env` files |
| `pandas`, `matplotlib` | Data analysis and visualization |

In [ ]:
# Install all required packages
!uv add \
    deepeval \
    ragas \
    qdrant-client \
    openai \
    langchain \
    langchain-openai \
    langchain-community \
    sentence-transformers \
    python-dotenv \
    pandas \
    matplotlib \
    tiktoken \
    numpy

After the install finishes you should see `Successfully installed ...` (or `Requirement already satisfied` if packages are present). If any errors appear, resolve them before continuing.

-
## 2. Load Environment Variables

We keep API keys in a `.env` file at the repository root so they are never committed to version control. The file should look like:

```
OPENAI_API_KEY=sk-...
```

We use `python-dotenv` to load these into the process environment.

In [ ]:
import os
from dotenv import load_dotenv

# Walk up from workbooks/ to the repo root to find .env
dotenv_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
if not os.path.exists(dotenv_path):
    dotenv_path = os.path.join(os.getcwd(), ".env")  # fallback: current dir

loaded = load_dotenv(dotenv_path, override=True)
print(f".env loaded from {dotenv_path}: {loaded}")

In [ ]:
# Verify that the critical keys are present
openai_key = os.getenv("OPENAI_API_KEY", "")

print(f"OPENAI_API_KEY  : {'set (' + openai_key[:8] + '...)' if openai_key else 'NOT SET'}")

if not openai_key:
    raise EnvironmentError(
        "OPENAI_API_KEY is not set. Create a .env file in the repo root with your key."
    )

-
## 3. Verify OpenAI API Key

We make a tiny API call to confirm the key is valid and the account has quota.

In [ ]:
from openai import OpenAI

client = OpenAI()  # picks up OPENAI_API_KEY automatically

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say 'hello' and nothing else."}],
    max_tokens=10,
)

print("OpenAI response:", response.choices[0].message.content)
print("Model used:     ", response.model)
print("\nOpenAI API key is valid and working.")

**Expected output:** You should see `hello` (or a close variant) printed above, along with the model name. If you get an `AuthenticationError`, double-check your key.

-
## 4. Test DeepEval Installation

DeepEval is the primary evaluation framework we will use. Let's confirm the import works and create a simple metric object.

In [ ]:
import deepeval

print(f"DeepEval version: {deepeval.__version__}")

In [ ]:
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

# Create a metric instance — this verifies the import chain works
metric = AnswerRelevancyMetric(
    threshold=0.7,
    model="gpt-4o-mini",
)
print(f"Metric created: {metric.__class__.__name__}")
print(f"Threshold:      {metric.threshold}")
print(f"Model:          {metric.model}")

In [ ]:
# Create a minimal test case
test_case = LLMTestCase(
    input="What is the capital of France?",
    actual_output="The capital of France is Paris.",
)
print(f"Test case created with input: '{test_case.input}'")
print(f"Actual output: '{test_case.actual_output}'")

If the cell above runs without errors, DeepEval is installed correctly. We will do a full evaluation in section 7.

-
## 5. Test RAGAS Installation

RAGAS is another popular evaluation framework for RAG pipelines. Let's verify the import chain.

In [ ]:
import ragas

print(f"RAGAS version: {ragas.__version__}")

In [ ]:
from ragas.metrics import Faithfulness, ResponseRelevancy
from ragas import SingleTurnSample, EvaluationDataset

print("Available RAGAS metrics imported successfully:")
print(f"  - Faithfulness")
print(f"  - ResponseRelevancy")
print(f"  - SingleTurnSample")
print(f"  - EvaluationDataset")

**Note:** RAGAS metric imports may vary across versions. If the import above fails, check the [RAGAS docs](https://docs.ragas.io/) for the correct import path for your version.

-
## 6. Test Qdrant Client Connection

Qdrant is the vector database we will use to store embeddings. We will use **in-memory mode** so no external server is required.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# Create an in-memory Qdrant instance
qdrant = QdrantClient(":memory:")
print(f"Qdrant client created (in-memory mode)")

# Create a small test collection
qdrant.create_collection(
    collection_name="test_collection",
    vectors_config=VectorParams(size=4, distance=Distance.COSINE),
)
print("Test collection created")

# Insert a point
qdrant.upsert(
    collection_name="test_collection",
    points=[
        PointStruct(id=1, vector=[0.1, 0.2, 0.3, 0.4], payload={"text": "hello world"}),
    ],
)
print("Test point inserted")

# Query
results = qdrant.query_points(
    collection_name="test_collection",
    query=[0.1, 0.2, 0.3, 0.4],
    limit=1,
)
print(f"Query result: {results}")
print("\nQdrant is working correctly.")

**Expected output:** You should see a query result with the point we just inserted, demonstrating that Qdrant's in-memory mode is fully functional.

-
## 7. Quick "Hello World" Evaluation — DeepEval

Let's run a real evaluation with DeepEval. We will use the `AnswerRelevancyMetric` to score whether an LLM output is relevant to the input question.

In [ ]:
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

# Create the metric
relevancy_metric = AnswerRelevancyMetric(
    threshold=0.7,
    model="gpt-4o-mini",
)

# Create a test case with a clearly relevant answer
good_test_case = LLMTestCase(
    input="What is photosynthesis?",
    actual_output="Photosynthesis is the process by which green plants convert sunlight, water, and carbon dioxide into glucose and oxygen. It primarily occurs in the chloroplasts of plant cells.",
)

# Measure
relevancy_metric.measure(good_test_case)

print(f"Score:  {relevancy_metric.score}")
print(f"Reason: {relevancy_metric.reason}")
print(f"Passed: {relevancy_metric.score >= relevancy_metric.threshold}")

**Expected output:** A score close to 1.0, because the answer is highly relevant to the question about photosynthesis.

In [ ]:
# Now test with an irrelevant answer
bad_test_case = LLMTestCase(
    input="What is photosynthesis?",
    actual_output="The stock market closed at 35,000 points today, showing a 2% increase from last week.",
)

relevancy_metric.measure(bad_test_case)

print(f"Score:  {relevancy_metric.score}")
print(f"Reason: {relevancy_metric.reason}")
print(f"Passed: {relevancy_metric.score >= relevancy_metric.threshold}")

**Expected output:** A score close to 0.0, because the answer about stock markets is completely irrelevant to a question about photosynthesis. This contrast demonstrates that the metric is working correctly.

-
## 8. Quick "Hello World" Evaluation — RAGAS

Now let's run a simple RAGAS evaluation. RAGAS expects a HuggingFace `Dataset` object with specific column names.

In [23]:
from ragas import evaluate
from ragas.metrics import Faithfulness, ResponseRelevancy
from ragas import EvaluationDataset, SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI

# Wrap the LLM for RAGAS
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))

# Create a single test sample
sample = SingleTurnSample(
    user_input="What is the capital of France?",
    response="The capital of France is Paris. It is located in the north-central part of the country along the Seine River.",
    retrieved_contexts=["Paris is the capital and most populous city of France. It is situated on the Seine River, in the north of the country."],
    reference="The capital of France is Paris.",
)

eval_dataset = EvaluationDataset(samples=[sample])
print("RAGAS EvaluationDataset created:")
print(f"  Samples: {len(eval_dataset.samples)}")

RAGAS EvaluationDataset created:
  Samples: 1


/var/folders/0l/l9k1zpt10x96skbkhhqnsr_w0000gp/T/ipykernel_90770/2928274695.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ResponseRelevancy
/var/folders/0l/l9k1zpt10x96skbkhhqnsr_w0000gp/T/ipykernel_90770/2928274695.py:2: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import Faithfulness, ResponseRelevancy
/var/folders/0l/l9k1zpt10x96skbkhhqnsr_w0000gp/T/ipykernel_90770/2928274695.py:8: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_f

In [24]:
# Run RAGAS evaluation
metrics = [Faithfulness(llm=evaluator_llm), ResponseRelevancy(llm=evaluator_llm)]

results = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
)

print("RAGAS Evaluation Results:")
results_df = results.to_pandas()
print(results_df)

Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

RAGAS Evaluation Results:
                       user_input  \
0  What is the capital of France?   

                                  retrieved_contexts  \
0  [Paris is the capital and most populous city o...   

                                            response  \
0  The capital of France is Paris. It is located ...   

                         reference  faithfulness  answer_relevancy  
0  The capital of France is Paris.      0.666667          0.949116  


**Expected output:** Both scores should be close to 1.0 because:
- The answer is faithful to the provided context (Paris is indeed stated as the capital).
- The answer is relevant to the question asked.

-
## 9. Additional Library Checks

Let's verify the remaining libraries we will use across the other notebooks.

In [25]:
import langchain
import sentence_transformers
import tiktoken
import pandas as pd
import matplotlib
import numpy as np

print(f"langchain:              {langchain.__version__}")
print(f"sentence-transformers:  {sentence_transformers.__version__}")
print(f"tiktoken:               {tiktoken.__version__}")
print(f"pandas:                 {pd.__version__}")
print(f"matplotlib:             {matplotlib.__version__}")
print(f"numpy:                  {np.__version__}")

langchain:              1.2.15
sentence-transformers:  5.4.0
tiktoken:               0.12.0
pandas:                 3.0.2
matplotlib:             3.10.8
numpy:                  2.4.4


-
## 10. Test OpenAI Embeddings

We will use OpenAI embeddings throughout the RAG pipeline. Let's confirm they work.

In [26]:
from openai import OpenAI

client = OpenAI()

embedding_response = client.embeddings.create(
    model="text-embedding-3-small",
    input="This is a test sentence for embedding.",
)

embedding = embedding_response.data[0].embedding
print(f"Embedding model: text-embedding-3-small")
print(f"Embedding dimension: {len(embedding)}")
print(f"First 10 values: {embedding[:10]}")
print("\nOpenAI embeddings are working correctly.")

Embedding model: text-embedding-3-small
Embedding dimension: 1536
First 10 values: [0.02081298828125, 0.00730133056640625, 0.00849151611328125, -0.016448974609375, -0.01227569580078125, -0.01548004150390625, 0.0294647216796875, -0.006069183349609375, 0.000713348388671875, 0.00882720947265625]

OpenAI embeddings are working correctly.


**Expected output:** An embedding vector of dimension 1536 (for `text-embedding-3-small`). The exact values will differ each time, but the dimensionality should be constant.

-
## 11. Test Cross-Encoder Reranker

We use a cross-encoder model from sentence-transformers for reranking. This runs locally — no API key needed.

In [27]:
from sentence_transformers import CrossEncoder

# Load a lightweight cross-encoder for reranking
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

query = "What is machine learning?"
documents = [
    "Machine learning is a subset of artificial intelligence.",
    "The weather today is sunny and warm.",
    "Deep learning uses neural networks with many layers.",
]

scores = reranker.predict([[query, doc] for doc in documents])

print("Cross-Encoder reranker results:")
for i, (doc, score) in enumerate(sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)):
    print(f"  Score: {score:.4f} | {doc}")

print("\nCross-encoder reranker is working correctly.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-Encoder reranker results:
  Score: 10.5139 | Machine learning is a subset of artificial intelligence.
  Score: -5.5301 | Deep learning uses neural networks with many layers.
  Score: -11.0817 | The weather today is sunny and warm.

Cross-encoder reranker is working correctly.


-
## 12. Environment Summary

Let's print a final summary of everything we verified.

In [28]:
import sys

print("=" * 60)
print("ENVIRONMENT SETUP SUMMARY")
print("=" * 60)
print(f"Python version:         {sys.version}")
print(f"OpenAI API key:         OK")
print(f"DeepEval:               OK")
print(f"RAGAS:                  OK")
print(f"Qdrant (in-memory):     OK")
print(f"OpenAI Embeddings:      OK")
print(f"Cross-Encoder Reranker: OK")
print(f"LangChain:              OK")
print("=" * 60)
print("\nAll systems are ready. Proceed to Notebook 02 to build the RAG pipeline.")

ENVIRONMENT SETUP SUMMARY
Python version:         3.13.9 (main, Nov 19 2025, 23:39:32) [Clang 21.1.4 ]
OpenAI API key:         OK
DeepEval:               OK
RAGAS:                  OK
Qdrant (in-memory):     OK
OpenAI Embeddings:      OK
Cross-Encoder Reranker: OK
LangChain:              OK

All systems are ready. Proceed to Notebook 02 to build the RAG pipeline.


-
## Next Steps

Now that the environment is configured and verified, proceed to:

- **Notebook 02** — Build a complete RAG pipeline with Qdrant, OpenAI, and cross-encoder reranking
- **Notebook 03** — Evaluate retriever quality with DeepEval metrics
- **Notebook 04** — Evaluate generator quality with DeepEval metrics
- **Notebook 05** — Advanced DeepEval features (G-Eval, custom metrics, synthetic data)